# PROJECT NOIR ? The Nishidak Broadcast
### Colab workprint renderer for the current 05:52 NOIR timeline
**Visual direction:** Sayan ? **Score:** `Noir.wav` by Atandra

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayan-bhattacharya/noir-analog-horror/blob/main/colab_noir_accelerator.ipynb)

This notebook clones the separate `noir-analog-horror` repository and renders its current timeline. The master screenplay and score are 30:22, while the implemented compositor currently ends at 05:52; this notebook renders that available workprint and does not imply the remaining runtime is implemented. It never substitutes podcast audio for `Noir.wav`.

Shot 30 includes slow courtyard-cloud motion, a mostly steady analog image, the purple charged-particle portal, and Bimal?s slow approach, slip, and fall.


## 1. Hardware Detection & Verification


In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[SUCCESS] GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[INFO] Running on Standard CPU. For fast renders, select Runtime -> Change runtime type -> T4 GPU")

## 1. Clone the standalone NOIR repository and install render dependencies


In [ ]:
import os, sys, subprocess
from pathlib import Path

project_dir = Path("/content/noir-analog-horror")
if not project_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/sayan-bhattacharya/noir-analog-horror.git", str(project_dir)], check=True)
else:
    print("Using existing NOIR checkout; no files will be deleted or reset.")
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
os.environ["PYTHONPATH"] = str(project_dir)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "soundfile", "scipy", "pillow", "uharfbuzz", "freetype-py", "pyyaml"], check=True)

# Use only the original NOIR score. First look in the mounted Drive location, then ask for an upload.
from google.colab import drive, files
drive.mount("/content/drive")
score_candidates = [
    Path("/content/drive/MyDrive/NOIR_Film/Noir.wav"),
    project_dir / "Noir.wav",
    project_dir / "audio" / "Noir.wav",
]
score_path = next((p for p in score_candidates if p.is_file()), None)
if score_path is None:
    print("Upload the original Noir.wav score (not a podcast mix).")
    uploaded = files.upload()
    score_path = next((Path(name) for name in uploaded if Path(name).name.lower() == "noir.wav"), None)
if score_path is None or not score_path.is_file():
    raise FileNotFoundError("Noir.wav is required. No fallback audio will be substituted.")
project_score = project_dir / "Noir.wav"
if score_path.resolve() != project_score.resolve():
    import shutil
    shutil.copy2(score_path, project_score)
print(f"NOIR score ready: {project_score}")


## 2. Render the current NOIR workprint
The renderer code currently covers `00:00?05:52` (352 seconds). The 30:22 screenplay has not yet been fully implemented in the compositor. Run a fast draft first; change `RENDER_QUALITY` to `final` after reviewing it.


In [ ]:
import os, sys, time, subprocess
from pathlib import Path

os.chdir("/content/noir-analog-horror")
RENDER_SECONDS = 352
RENDER_QUALITY = "preview"  # change to "final" after reviewing the draft
cmd = [sys.executable, "-m", "noir_film.render_noir", "--duration", str(RENDER_SECONDS), "--quality", RENDER_QUALITY]
print("Rendering current NOIR workprint:", " ".join(cmd))
start_time = time.time()
subprocess.run(cmd, check=True)
print(f"Render complete in {time.time() - start_time:.1f}s")


## 4. In-Notebook Video Player


In [ ]:
from IPython.display import Video, display
from pathlib import Path
video_path = Path("/content/noir-analog-horror/output/noir_master_352s_preview.mp4")
if video_path.is_file():
    display(Video(str(video_path), embed=False, width=900))
else:
    raise FileNotFoundError(f"Expected NOIR render not found: {video_path}")


## 5. Download Master Film (720p HD MP4)


In [ ]:
from google.colab import files
from pathlib import Path
video_path = Path("/content/noir-analog-horror/output/noir_master_352s_preview.mp4")
if not video_path.is_file():
    raise FileNotFoundError(f"Expected NOIR render not found: {video_path}")
files.download(str(video_path))


## 4. Save the NOIR workprint to Google Drive


In [ ]:
import shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
src = Path("/content/noir-analog-horror/output/noir_master_352s_preview.mp4")
dest_dir = Path("/content/drive/MyDrive/NOIR_Film/renders")
dest_dir.mkdir(parents=True, exist_ok=True)
if not src.is_file():
    raise FileNotFoundError(f"Expected NOIR render not found: {src}")
dest = dest_dir / src.name
shutil.copy2(src, dest)
print(f"Saved NOIR workprint: {dest}")
